<a href="https://colab.research.google.com/github/manek1/de_da_traning/blob/feature_ashwathi/Eligibility_Spark_code_using_DataframeAPI_in_class_methods_format.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, lit, greatest, least
from pyspark.sql.types import TimestampType

# Initialize Spark session
spark = SparkSession.builder.appName("EligibilityProcessor").getOrCreate()

class EligibilityProcessor:
    def __init__(self, spark):
        self.spark = spark

    def load_data(self):
        # Load CSVs as DataFrames
        self.client = self.spark.read.option("header", True).csv("/content/client.csv")
        self.policy = self.spark.read.option("header", True).csv("/content/policy.csv")
        self.clientcontract = self.spark.read.option("header", True).csv("/content/clientcontract.csv")
        self.countryofresidence = self.spark.read.option("header", True).csv("/content/countryofresidence.csv")
        self.membercoverage = self.spark.read.option("header", True).csv("/content/membercoverage.csv")
        self.groupcoverage = self.spark.read.option("header", True).csv("/content/groupcoverage.csv")
        self.legalentity = self.spark.read.option("header", True).csv("/content/legalentity.csv")
        self.lineofbusiness = self.spark.read.option("header", True).csv("/content/lineofbusiness.csv")
        self.eligibility = self.spark.read.option("header", True).csv("/content/eligibility.csv")
        self.member = self.spark.read.option("header", True).csv("/content/member.csv")
        self.customer = self.spark.read.option("header", True).csv("/content/customer.csv")
        self.clientstaff_category = self.spark.read.option("header", True).csv("/content/clientstaff_category.csv")

    def preprocess_data(self):
        # Cast timestamp fields for countryofresidence and eligibility DataFrames
        self.countryofresidence = self.countryofresidence.withColumn("countryofresidencestartdate",
                                                                    coalesce(col("countryofresidencestartdate"), lit("1900-01-01 00:00:00")).cast(TimestampType()))
        self.countryofresidence = self.countryofresidence.withColumn("countryofresidenceenddate",
                                                                    coalesce(col("countryofresidenceenddate"), lit("9999-12-31 00:00:00")).cast(TimestampType()))
        self.eligibility = self.eligibility.withColumn("eligibilityfromdate", col("eligibilityfromdate").cast(TimestampType()))
        self.eligibility = self.eligibility.withColumn("eligibilitytodate", col("eligibilitytodate").cast(TimestampType()))

    def create_cli(self):
        cli = self.client.filter(col("currentrecordindicator") == "true") \
            .join(self.policy.filter(col("currentrecordindicator") == "true"),
                  (self.client["clientid"] == self.policy["clientid"]) &
                  (self.client["sourcesystemid"] == self.policy["sourcesystemid"])) \
            .join(self.clientcontract,
                  (self.client["clientcontractid"] == self.clientcontract["clientcontractid"]) &
                  (self.client["sourcesystemid"] == self.clientcontract["sourcesystemid"]),
                  "left") \
            .select(
                self.client["clientid"].alias("clientid"),
                self.client["clientname"].alias("clientname"),
                self.client["sourcesystemid"].alias("sourcesystemid"),
                self.policy["policyid"].alias("policyid"),
                self.policy["policyname"].alias("policyname"),
                self.client["clientcontractid"].alias("clientcontractid"),
                self.policy["premiumcurrencycode"].alias("premiumcurrencycode"),
                self.client["PricingMethodCode"].alias("PricingMethodCode"),
                self.client["PremiumFundingArrangementCode"].alias("PremiumFundingArrangementCode"),
                coalesce(col("clientcontractstartdate"), lit(None).cast("string")).alias("clientcontractstartdate"),
                coalesce(col("clientcontractenddate"), lit(None).cast("string")).alias("clientcontractenddate"),
                self.client["clientinceptiondate"].alias("clientinceptiondate"),
                self.client["clientterminationdate"].alias("clientterminationdate"),
                self.client["clientbusinesssegmentcode"].alias("clientbusinesssegmentcode")
            )
        return cli

    def create_res_country(self):
        res_country = self.countryofresidence \
            .filter((col("currentrecordindicator") == "true") &
                    (col("countryofresidencetypecode") == "ASS") &
                    ~(col("countryofresidenceenddate").isNull() & col("countryofresidencestartdate").isNull())) \
            .selectExpr("countryofresidencecode", "customerid",
                        "cast(countryofresidenceenddate as timestamp) as countryofresidenceenddate",
                        "cast(countryofresidencestartdate as timestamp) as countryofresidencestartdate",
                        "countryofresidenceenddate as cor_end_date",
                        "sourcesystemid", "etlchecksum as cor_checksum") \
            .distinct()
        return res_country

    def create_coverage(self):
        coverage = self.membercoverage.filter(col("currentrecordindicator") == "true") \
            .join(self.groupcoverage.filter(col("currentrecordindicator") == "true"),
                  (self.membercoverage["groupcoverageskey"] == self.groupcoverage["groupcoverageskey"]) &
                  (self.membercoverage["sourcesystemid"] == self.groupcoverage["sourcesystemid"]) &
                  (self.membercoverage["areaofcovercode"] == self.groupcoverage["areaofcovercode"]) &
                  (self.membercoverage["relationshiptypecode"] == self.groupcoverage["relationshiptypecode"]) &
                  (self.membercoverage["policyid"] == self.groupcoverage["policyid"]) &
                  (self.membercoverage["planid"] == self.groupcoverage["planid"]) &
                  (self.membercoverage["packageid"] == self.groupcoverage["packageid"]) &
                  (self.membercoverage["clientstaff categoryid"] == self.groupcoverage["clientstaff categoryid"])) \
            .join(self.legalentity.filter(col("currentrecordindicator") == "true"),
                  (self.groupcoverage["primarylegalentityid"] == self.legalentity["legalentityid"]) &
                  (self.groupcoverage["sourcesystemid"] == self.legalentity["sourcesystemid"])) \
            .join(self.lineofbusiness,
                  self.legalentity["lineofbusinessid"] == self.lineofbusiness["lineofbusinessid"],
                  "left") \
            .select(
                self.groupcoverage["primarylegalentityid"], self.groupcoverage["medicalunderwritingcustomertypecode"],
                self.membercoverage["memberid"], self.membercoverage["policyid"], self.membercoverage["planid"],
                self.membercoverage["packageid"], self.membercoverage["sourcesystemid"],
                self.legalentity["legalentityname"], self.lineofbusiness["lineofbusinessname"]
            )
        return coverage

    def create_elg(self, coverage):
        memb = self.member.filter(col("currentrecordindicator") == "true") \
            .join(self.customer.filter(col("currentrecordindicator") == "true"),
                  (self.member["customerid"] == self.customer["customerid"]) &
                  (self.member["sourcesystemid"] == self.customer["sourcesystemid"])) \
            .join(self.clientstaff_category.filter(col("currentrecordindicator") == "true"),
                  self.member["clientstaff categoryid"] == self.clientstaff_category["clientstaff categoryid"]) \
            .select(
                self.member["memberid"], self.customer["dateofbirth"], self.customer["customerid"],
                self.member["clientstaff categoryid"].alias("clientstaff_categoryid"),
                self.clientstaff_category["clientstaff categoryname"].alias("clientstaff_categoryname"),
                self.member["sourcesystemid"]
            )

        elg = self.eligibility.join(memb,
                                    (self.eligibility["memberid"] == memb["memberid"]) &
                                    (self.eligibility["sourcesystemid"] == memb["sourcesystemid"])) \
            .join(coverage,
                  (self.eligibility["memberid"] == coverage["memberid"]) &
                  (self.eligibility["policyid"] == coverage["policyid"]) &
                  (self.eligibility["planid"] == coverage["planid"]) &
                  (self.eligibility["sourcesystemid"] == coverage["sourcesystemid"])) \
            .select(
                self.eligibility["planid"], self.eligibility["etlchecksum"].alias("elg_checksum"),
                self.eligibility["areaofcovercode"], self.eligibility["relationshiptypecode"],
                self.eligibility["memberid"], self.eligibility["familyunittypecode"], self.eligibility["policyid"],
                memb["dateofbirth"], memb["customerid"], self.eligibility["premiumloadingpercentage"],
                coverage["primarylegalentityid"], coverage["medicalunderwritingcustomertypecode"],
                coverage["legalentityname"], coverage["lineofbusinessname"], coverage["packageid"],
                memb["clientstaff_categoryid"], memb["clientstaff_categoryname"],
                self.eligibility["eligibilityfromdate"], self.eligibility["eligibilitytodate"],
                self.eligibility["sourcesystemid"]
            )
        return elg

    def create_final_result(self, cli, res_country, elg):
        final_df = elg.join(cli,
                            (elg["policyid"] == cli["policyid"]) &
                            (elg["sourcesystemid"] == cli["sourcesystemid"])) \
            .join(res_country,
                  (elg["customerid"] == res_country["customerid"]) &
                  (elg["sourcesystemid"] == res_country["sourcesystemid"]) &
                  (
                          (elg["eligibilityfromdate"].between(res_country["countryofresidencestartdate"], res_country["countryofresidenceenddate"])) |
                          (res_country["countryofresidencestartdate"].between(elg["eligibilityfromdate"], elg["eligibilitytodate"]))
                  ), "left") \
            .filter(col("memberid").isin("05044007901", "85000435704", "20026979201", "85100917702")) \
            .select(
                elg["customerid"], elg["memberid"], cli["clientid"], cli["clientname"],
                cli["clientcontractstartdate"], cli["clientcontractenddate"],
                cli["clientinceptiondate"], cli["clientterminationdate"],
                cli["clientbusinesssegmentcode"], cli["PremiumFundingArrangementCode"],
                cli["PricingMethodCode"], elg["policyid"], cli["policyname"],
                cli["premiumcurrencycode"], elg["planid"], elg["packageid"],
                elg["clientstaff_categoryid"], elg["clientstaff_categoryname"],
                elg["primarylegalentityid"], elg["legalentityname"], elg["lineofbusinessname"],
                elg["areaofcovercode"], elg["relationshiptypecode"], elg["familyunittypecode"],
                elg["medicalunderwritingcustomertypecode"], elg["premiumloadingpercentage"],
                elg["eligibilityfromdate"], elg["eligibilitytodate"],
                res_country["countryofresidencestartdate"], res_country["countryofresidenceenddate"],
                res_country["countryofresidencecode"],
                greatest(coalesce(res_country["countryofresidencestartdate"], lit("1900-01-01 00:00:00").cast(TimestampType())),
                         elg["eligibilityfromdate"]).alias("record_from_date"),
                least(coalesce(res_country["countryofresidenceenddate"], lit("9999-12-31 00:00:00").cast(TimestampType())),
                      elg["eligibilitytodate"]).alias("record_to_date"),
                elg["dateofbirth"], elg["elg_checksum"], res_country["cor_checksum"]
            )
        return final_df

    def run(self):
        self.load_data()
        self.preprocess_data()

        cli = self.create_cli()
        res_country = self.create_res_country()
        coverage = self.create_coverage()
        elg = self.create_elg(coverage)
        final_result = self.create_final_result(cli, res_country, elg)

        # Show the result
        final_result.show(truncate=False)



# Instantiate the EligibilityProcessor and run the transformation
processor = EligibilityProcessor(spark)
processor.run()


+----------+--------+--------+----------+-----------------------+---------------------+-------------------+---------------------+-------------------------+-----------------------------+-----------------+--------+----------+-------------------+------+---------+----------------------+------------------------+--------------------+---------------+------------------+---------------+--------------------+------------------+-----------------------------------+------------------------+-------------------+-----------------+---------------------------+-------------------------+----------------------+----------------+--------------+-----------+------------+------------+
|customerid|memberid|clientid|clientname|clientcontractstartdate|clientcontractenddate|clientinceptiondate|clientterminationdate|clientbusinesssegmentcode|PremiumFundingArrangementCode|PricingMethodCode|policyid|policyname|premiumcurrencycode|planid|packageid|clientstaff_categoryid|clientstaff_categoryname|primarylegalentityid|lega